# Chakravyuh — v2 LoRA per-row eval + leakage-clean slice (B.12)

**What this notebook does (one Colab T4 unit, ~6 min runtime):**

1. Loads `ujjwalpardeshi/chakravyuh-analyzer-lora-v2` over Qwen2.5-7B-Instruct (4-bit).
2. Scores all 174 scenarios in `data/chakravyuh-bench-v0/scenarios.jsonl` using the **exact training prompt format** (via `LLMAnalyzer.build_prompt`).
3. Joins per-row scores with `logs/semantic_leakage_audit.json` to get `max_cosine_to_training` per scenario.
4. Computes detection / FPR / F1 on the **leakage-clean subset** (cosine < 0.70: 38 scams + 12 benigns = 50 scenarios).
5. Saves three artifacts:
   - `logs/eval_v2_per_row.jsonl` — per-scenario {id, score, predicted, ground_truth, leakage_cosine}
   - `logs/eval_v2_reproduce.json` — aggregate metrics (matches existing schema)
   - `docs/leakage_clean_eval.md` — markdown report ready to commit

**Use case:** unblocks B.6 (calibration), D.5 (failure taxonomy), B.17 (cheap-eval weight grid), and produces the leakage-clean OOD number for the README + Q&A 3b.

**Runtime:** Colab T4 (free / 1 unit). A100 if available is faster but unnecessary.

## 1. Verify GPU + runtime

In [ ]:
import subprocess, sys

print('Python:', sys.version)
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader']).decode().strip()
    print('GPU:', out)
except Exception as e:
    raise RuntimeError(f'No GPU detected. Switch Colab runtime to T4/L4/A100. Error: {e}')

## 2. Clone repo + install dependencies

We pin versions to match what the v2 LoRA was trained against. Do not let Colab silently upgrade transformers > 4.46 — older PEFT adapters can break.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/UjjwalPardeshi/Chakravyuh'
REPO_DIR = '/content/Chakravyuh'

if not Path(REPO_DIR).exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --rebase --autostash

%cd {REPO_DIR}
!git rev-parse HEAD

In [ ]:
# Editable install + clean ML-stack reinstall + KERNEL RESTART.
#
# CRITICAL: Colab pre-imports transformers 4.42.x (without SlidingWindowCache).
# Even after `pip install transformers==4.46.3`, the running Python kernel still
# holds the OLD modules in sys.modules. The only reliable fix is to KILL the
# kernel after install, then re-run from cell 4 onwards.
#
# This cell will:
#   1. Install Chakravyuh + pinned transformers/peft/accelerate/bitsandbytes
#   2. Trigger a kernel restart (you'll see 'Your session crashed' — that's normal)
#   3. After restart, run cells 4 (clone, idempotent) and 6 onwards. SKIP CELL 5 ON RE-RUN.

%cd {REPO_DIR}

# Step 1: install Chakravyuh package itself (no deps)
!pip install -q -e '.[llm]' --no-deps 2>&1 | tail -3

# Step 2: nuke any pre-existing ML packages
!pip uninstall -y -q transformers peft accelerate bitsandbytes 2>&1 | tail -3

# Step 3: clean reinstall of pinned versions
!pip install -q --no-cache-dir --force-reinstall \
    'transformers==4.46.3' \
    'peft==0.13.2' \
    'accelerate==1.0.1' \
    'bitsandbytes==0.44.1' \
    2>&1 | tail -5

# Step 4: small extras
!pip install -q 'sentencepiece' 'pydantic>=2.0' 'tqdm' 2>&1 | tail -3

# Step 5: verify packages are on disk (this does NOT prove the kernel sees them)
!python -c "import transformers; print('transformers on disk:', transformers.__version__)"
!python -c "from transformers.cache_utils import SlidingWindowCache; print('SlidingWindowCache symbol: OK')"

# Step 6: KILL THE KERNEL. After Colab auto-reconnects, run cells 4, 6, 8, 9... (SKIP cell 5).
import os
print('\n' + '=' * 60)
print('INSTALL COMPLETE. Restarting kernel in 2 seconds...')
print('AFTER RESTART:')
print('  1. Re-run cell 4 (re-clone — idempotent, fast)')
print('  2. Skip cell 5 (already done)')
print('  3. Run cells 6 onwards normally')
print('=' * 60)
import time; time.sleep(2)
os.kill(os.getpid(), 9)  # hard kill — Colab will auto-reconnect

In [ ]:
# We deliberately SKIP Unsloth on Colab.
# Reason: Unsloth's torch shim breaks torchvision's ABI on Colab T4, which then
# fails when transformers tries to import Qwen2ForCausalLM (circular torchvision import).
# The plain transformers + bitsandbytes 4-bit path is ~30% slower but reliable.
# Total runtime impact: +2 min on a 174-scenario eval. Acceptable.
print('Unsloth intentionally skipped on Colab — using transformers + bitsandbytes 4-bit fallback.')

## 3. Load the v2 LoRA

Uses the existing `LLMAnalyzer` class — same code path as training and the live HF Space. This guarantees the inference prompt **exactly matches** what GRPO was trained against.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from chakravyuh_env.agents.llm_analyzer import LLMAnalyzer

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
LORA_ADAPTER = 'ujjwalpardeshi/chakravyuh-analyzer-lora-v2'

analyzer = LLMAnalyzer(
    model_name=BASE_MODEL,
    lora_path=LORA_ADAPTER,
    use_unsloth=False,        # CRITICAL: skip unsloth on Colab — it corrupts torchvision
    load_in_4bit=True,
    max_new_tokens=200,
    temperature=0.0,
    flag_threshold=0.55,
    seed=42,
)
analyzer.load()  # ~3-4 min on T4 (downloads + materializes 4-bit weights + LoRA)
print('Analyzer ready. Threshold =', analyzer.flag_threshold)

In [ ]:
# Sanity check: a known scam should score >= 0.7, a known benign <= 0.3.
smoke_scam = 'URGENT: Your SBI account will be blocked today. Share OTP 234567 to verify KYC.'
smoke_benign = 'Your OTP for HDFC login is 563182. Valid 5 min. Do NOT share with anyone.'

print('Scam test  :', analyzer.score_text(smoke_scam))
print('Benign test:', analyzer.score_text(smoke_benign))

## 4. Load bench scenarios + leakage audit

In [ ]:
import json
from pathlib import Path

BENCH_PATH = Path(REPO_DIR) / 'data' / 'chakravyuh-bench-v0' / 'scenarios.jsonl'
LEAKAGE_PATH = Path(REPO_DIR) / 'logs' / 'semantic_leakage_audit.json'

with BENCH_PATH.open() as f:
    scenarios = [json.loads(line) for line in f if line.strip()]

with LEAKAGE_PATH.open() as f:
    leakage = json.load(f)

leakage_by_id = {row['scenario_id']: row for row in leakage['per_scenario']}

print(f'Loaded {len(scenarios)} bench scenarios.')
print(f'Loaded {len(leakage_by_id)} leakage-audit rows (cosine to nearest training neighbor).')
print(f'Bench scenarios with leakage data: {sum(1 for s in scenarios if s["id"] in leakage_by_id)}')

## 5. Score every scenario with v2 LoRA

~2 sec/scenario on T4 → ~6 min for 174 scenarios. Idempotent: if you re-run, results are deterministic (`temperature=0.0, seed=42`).

In [ ]:
from tqdm.auto import tqdm
import time

THRESHOLD = analyzer.flag_threshold  # 0.55, matches training

per_row = []
errors = []
t0 = time.time()

for s in tqdm(scenarios, desc='scoring'):
    sid = s['id']
    truth = s['ground_truth']['is_scam']
    difficulty = s['ground_truth']['difficulty']
    category = s['ground_truth']['category']

    # Concatenate all scammer turns (matches mode_c_real_cases.py + training).
    text = ' '.join(
        msg['text'] for msg in s['attack_sequence'] if msg['sender'] == 'scammer'
    )

    try:
        score = analyzer.score_text(text)
    except Exception as e:
        errors.append({'scenario_id': sid, 'error': str(e)})
        continue

    predicted_flag = score >= THRESHOLD
    leak_row = leakage_by_id.get(sid, {})

    per_row.append({
        'scenario_id': sid,
        'score': float(score),
        'predicted_flag': bool(predicted_flag),
        'ground_truth': bool(truth),
        'correct': bool(predicted_flag) == bool(truth),
        'difficulty': difficulty,
        'category': category,
        'leakage_cosine': leak_row.get('max_cosine_to_training'),
        'nearest_template_id': leak_row.get('nearest_template_id'),
    })

elapsed = time.time() - t0
print(f'\nScored {len(per_row)} / {len(scenarios)} in {elapsed/60:.1f} min ({elapsed/len(scenarios):.2f} sec/row).')
if errors:
    print(f'Skipped {len(errors)} scenarios due to inference error:')
    for e in errors[:5]:
        print('  -', e)

## 6. Compute aggregate metrics + per-difficulty breakdown

In [ ]:
from collections import defaultdict

def aggregate(rows):
    n = len(rows)
    if not n:
        return {'n': 0}
    tp = sum(1 for r in rows if r['ground_truth'] and r['predicted_flag'])
    fn = sum(1 for r in rows if r['ground_truth'] and not r['predicted_flag'])
    fp = sum(1 for r in rows if not r['ground_truth'] and r['predicted_flag'])
    tn = sum(1 for r in rows if not r['ground_truth'] and not r['predicted_flag'])
    scam = tp + fn
    benign = fp + tn
    detection = tp / scam if scam else 0.0
    fpr = fp / benign if benign else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = (2 * precision * detection / (precision + detection)) if (precision + detection) else 0.0
    return {
        'n': n,
        'n_scam': scam,
        'n_benign': benign,
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'detection': detection,
        'fpr': fpr,
        'precision': precision,
        'f1': f1,
        'accuracy': (tp + tn) / n,
    }

overall = aggregate(per_row)
by_difficulty = {}
buckets = defaultdict(list)
for r in per_row:
    buckets[r['difficulty']].append(r)
for diff, rows in buckets.items():
    by_difficulty[diff] = aggregate(rows)

print('=== Overall (in-distribution, n =', overall['n'], ') ===')
print(f"detection : {overall['detection']:.1%}")
print(f"FPR       : {overall['fpr']:.1%}")
print(f"F1        : {overall['f1']:.4f}")
print(f"precision : {overall['precision']:.1%}")
print(f"accuracy  : {overall['accuracy']:.1%}")
print()
print('=== Per-difficulty ===')
for diff in ('easy', 'medium', 'hard', 'novel'):
    if diff in by_difficulty:
        m = by_difficulty[diff]
        print(f"  {diff:8s} n={m['n']:3d}  detection={m['detection']:.1%}  fpr={m['fpr']:.1%}")

## 7. Compute leakage-clean slice (cosine < 0.70) — the OOD-honest number

This is the headline number for Q&A 3b: *"What's your real generalization number?"*

In [ ]:
LEAKAGE_THRESHOLDS = [0.50, 0.70, 0.85, 0.95]

slice_metrics = {}
for thr in LEAKAGE_THRESHOLDS:
    clean = [r for r in per_row if r['leakage_cosine'] is not None and r['leakage_cosine'] < thr]
    slice_metrics[thr] = aggregate(clean)

print('Leakage-clean slices (lower cosine = more OOD):')
print(f"{'cosine_lt':>12s} {'n':>4s} {'n_scam':>7s} {'n_benign':>9s} {'detection':>10s} {'fpr':>8s} {'f1':>8s}")
for thr in LEAKAGE_THRESHOLDS:
    m = slice_metrics[thr]
    if m['n'] == 0:
        print(f'{thr:12.2f}   (no scenarios pass this threshold)')
        continue
    print(f"{thr:12.2f} {m['n']:4d} {m['n_scam']:7d} {m['n_benign']:9d} {m['detection']:10.1%} {m['fpr']:8.1%} {m['f1']:8.4f}")

# Headline OOD slice (the one limitations.md references):
honest = slice_metrics[0.70]
if honest['n'] > 0:
    print()
    print('=== HONEST OOD HEADLINE (cosine < 0.70, n=', honest['n'], ') ===')
    print(f"  detection : {honest['detection']:.1%}")
    print(f"  FPR       : {honest['fpr']:.1%}")
    print(f"  F1        : {honest['f1']:.4f}")
    print()
    print('Memorize this for Q&A 3b. Cite it in the README + slides.')

## 8. Save artifacts (3 files)

In [ ]:
from pathlib import Path

LOGS = Path(REPO_DIR) / 'logs'
DOCS = Path(REPO_DIR) / 'docs'
LOGS.mkdir(exist_ok=True)
DOCS.mkdir(exist_ok=True)

# 8a. Per-row JSONL — used by D.5 failure taxonomy + B.17 weight grid + B.6 calibration.
per_row_path = LOGS / 'eval_v2_per_row.jsonl'
with per_row_path.open('w') as f:
    for r in per_row:
        f.write(json.dumps(r) + '\n')

# 8b. Aggregate JSON — same schema as the existing logs/eval_v2.json so downstream tools just work.
agg_path = LOGS / 'eval_v2_reproduce.json'
agg_payload = {
    'lora_v2': {
        'detection': overall['detection'],
        'fpr': overall['fpr'],
        'precision': overall['precision'],
        'f1': overall['f1'],
        'n': overall['n'],
        'per_difficulty': {
            d: {'n': m['n'], 'detection_rate': m['detection']}
            for d, m in by_difficulty.items()
        },
        'threshold': THRESHOLD,
    },
    'leakage_clean_slices': {
        f'cosine_lt_{thr:.2f}': slice_metrics[thr]
        for thr in LEAKAGE_THRESHOLDS
    },
    'meta': {
        'base_model': BASE_MODEL,
        'lora_adapter': LORA_ADAPTER,
        'threshold': THRESHOLD,
        'temperature': 0.0,
        'seed': 42,
        'n_attempted': len(scenarios),
        'n_scored': len(per_row),
        'errors': len(errors),
    },
}
with agg_path.open('w') as f:
    json.dump(agg_payload, f, indent=2)

# 8c. Markdown report — paste straight into limitations.md / README.
md_lines = [
    '# v2 LoRA — leakage-clean evaluation\n',
    '_Auto-generated by `notebooks/v2_per_row_eval_colab.ipynb`. Source artifact: `logs/eval_v2_per_row.jsonl`._\n',
    f'\n**Base model:** `{BASE_MODEL}`  ',
    f'\n**LoRA adapter:** `{LORA_ADAPTER}`  ',
    f'\n**Threshold:** {THRESHOLD}  ',
    f'\n**Seed:** 42  &nbsp;**Temperature:** 0.0\n',
    '\n## Headline numbers\n',
    '| Slice | n | n_scam | n_benign | Detection | FPR | F1 |',
    '|---|---:|---:|---:|---:|---:|---:|',
    f"| **Full bench (in-distribution)** | {overall['n']} | {overall['n_scam']} | {overall['n_benign']} | {overall['detection']:.1%} | {overall['fpr']:.1%} | {overall['f1']:.4f} |",
]
for thr in LEAKAGE_THRESHOLDS:
    m = slice_metrics[thr]
    if m['n'] == 0:
        continue
    label = f'Leakage-clean (cosine < {thr:.2f})'
    if abs(thr - 0.70) < 1e-9:
        label = f'**{label}** ⭐ honest OOD'
    md_lines.append(
        f"| {label} | {m['n']} | {m['n_scam']} | {m['n_benign']} | {m['detection']:.1%} | {m['fpr']:.1%} | {m['f1']:.4f} |"
    )

md_lines += [
    '\n## Per-difficulty (full bench)',
    '| Difficulty | n | Detection |',
    '|---|---:|---:|',
]
for diff in ('easy', 'medium', 'hard', 'novel'):
    if diff in by_difficulty:
        m = by_difficulty[diff]
        md_lines.append(f"| {diff} | {m['n']} | {m['detection']:.1%} |")

md_lines += [
    '\n## Reading guide\n',
    '- The **full-bench number** is in-distribution per the semantic-leakage audit (44.8% of bench has cosine > 0.85 to training). The 100% on easy/medium/hard is partly memorization-driven.',
    '- The **leakage-clean (cosine < 0.70) number** is the closest measurable approximation to true OOD generalization. Cite this for any "real generalization" question.',
    '- The **v1 → v2 FPR collapse** (36% → 6.7%) is unaffected by leakage because both versions evaluated on the same bench substrate.',
]
md_path = DOCS / 'leakage_clean_eval.md'
with md_path.open('w') as f:
    f.write('\n'.join(md_lines) + '\n')

print('Wrote:')
print(' -', per_row_path)
print(' -', agg_path)
print(' -', md_path)

## 9. Download artifacts

In [ ]:
# Colab download. After this runs, copy the three files into your repo, commit + push.
try:
    from google.colab import files
    for p in (per_row_path, agg_path, md_path):
        files.download(str(p))
except ImportError:
    print('Not running in Colab — skip download. Files are at:')
    for p in (per_row_path, agg_path, md_path):
        print(' ', p)

## 10. What to do with these files

1. Copy `logs/eval_v2_per_row.jsonl` and `logs/eval_v2_reproduce.json` into your local repo's `logs/` folder.
2. Copy `docs/leakage_clean_eval.md` into your local repo's `docs/` folder.
3. Update `docs/limitations.md` — replace the placeholder "v3 work — flagged below" line with the actual leakage-clean detection / FPR / F1 numbers from this notebook.
4. Update `README.md` — add a one-line caveat in the Limitations section: *"Leakage-clean (cosine < 0.70, n=50) detection is X%, FPR Y%; full-bench numbers are in-distribution per the semantic-leakage audit."*
5. Update `docs/Q_AND_A_REHEARSAL.md` Q3b — replace "is v3 work" with the measured number.
6. Commit + push.

**Subsequent unblocked items (use the same per-row JSONL):**
- B.6 calibration ECE + reliability diagram
- B.17 cheap-eval weight grid (re-score under different reward weights — 0 GPU)
- D.5 failure-mode taxonomy (group the 2 FPs + 1 missed scam)
- E.4 case-study writeups (3 scenarios — pick from the per-row data)